# BirdCLEF 2026 — Perch baseline (ONNX runtime, 1536-d embeddings)

Uses ONNX runtime instead of TF SavedModel. ONNX is dramatically faster on M1 and avoids the XLA compilation hang.

**Key wins over BirdNET:**
1. Perch embeddings (1536-d) are stronger for bird audio
2. Direct logit mapping for ~70% of species (free signal from Perch's built-in classifier)
3. Genus proxy for the rest
4. Same residual MLP head architecture as the best BirdNET head

**Setup before running:**
Make sure you've downloaded the ONNX model. Run this once in a terminal or in a code cell:
```python
import kagglehub
path = kagglehub.dataset_download('rishikeshjani/perch-onnx-for-birdclef-2026')
print(path)
```
Then update `PERCH_ONNX_PATH` below to point at the `perch_v2.onnx` file inside that folder.

## Stage 0 — Install ONNX runtime

In [1]:
%pip install -q onnxruntime kagglehub librosa scikit-learn tensorflow tqdm

Note: you may need to restart the kernel to use updated packages.


## Stage 0b — Download Perch ONNX model (one-time)

In [2]:
import kagglehub
from pathlib import Path

_perch_onnx_dir = Path(kagglehub.dataset_download('rishikeshjani/perch-onnx-for-birdclef-2026'))
print('Downloaded to:', _perch_onnx_dir)
print('\nFiles in dataset:')
for f in sorted(_perch_onnx_dir.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(_perch_onnx_dir)}  ({f.stat().st_size/1e6:.1f} MB)')

# Find the .onnx file
_onnx_files = list(_perch_onnx_dir.rglob('*.onnx'))
if not _onnx_files:
    raise FileNotFoundError('No .onnx file found in dataset')
PERCH_ONNX_PATH = _onnx_files[0]
print(f'\nUsing ONNX file: {PERCH_ONNX_PATH}')

Downloaded to: /Users/enricozafiris/.cache/kagglehub/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/versions/2

Files in dataset:
  labels.csv  (0.3 MB)
  onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl  (17.2 MB)
  perch_v2.onnx  (409.1 MB)

Using ONNX file: /Users/enricozafiris/.cache/kagglehub/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/versions/2/perch_v2.onnx


## Imports, paths, constants

In [3]:
from __future__ import annotations
import gc, json, re, time, warnings
from pathlib import Path

import librosa
import numpy as np
import onnxruntime as ort
import pandas as pd
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

# TF only used to train the head (small, no XLA issues)
import tensorflow as tf
try:
    tf.config.set_visible_devices([], 'GPU')
except Exception:
    pass

warnings.filterwarnings('ignore', category=UserWarning)

# ── Absolute paths (as you requested) ────────────────────────────────────
PROJECT_ROOT          = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project')
DATA_DIR              = PROJECT_ROOT / 'data'
TRAIN_AUDIO_DIR       = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/data/train_audio')
TRAIN_SOUNDSCAPES_DIR = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/data/train_soundscapes')
TRAIN_CSV             = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/data/train.csv')
LABELS_CSV            = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/data/train_soundscapes_labels.csv')
SAMPLE_SUB_CSV        = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/data/sample_submission.csv')
TAXONOMY_CSV          = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/data/taxonomy.csv')

CACHE_DIR     = PROJECT_ROOT / 'notebooks' / 'perch_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NPZ_VAL       = CACHE_DIR / 'perch_val_emb1536.npz'
NPZ_TRAIN     = CACHE_DIR / 'perch_train_emb1536.npz'

AUTOSAVE_ROOT     = PROJECT_ROOT / 'notebooks' / 'submission_perch_pseudo_labeling'
AUTOSAVE_RUN_NAME = 'submission_perch_baseline'
AUTOSAVE_DIR      = AUTOSAVE_ROOT / AUTOSAVE_RUN_NAME
AUTOSAVE_DIR.mkdir(parents=True, exist_ok=True)

# Perch labels (for direct logit mapping). Comes from the official TF SavedModel
# downloaded by kagglehub.model_download. We only need the labels.csv from there.
PERCH_LABELS_CSV = Path(
    '/Users/enricozafiris/.cache/kagglehub/models/'
    'google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu/1/'
    'assets/labels.csv'
)

# ── Audio constants ──────────────────────────────────────────────────────
SR_LOAD       = 32_000
PERCH_SR      = 32_000
WINDOW_SEC    = 5.0
N_WINDOWS     = 12
EMB_DIM       = 1536
PERCH_SAMPLES = int(WINDOW_SEC * PERCH_SR)   # 160_000

MIX_PROB    = 0.50
SNR_MIN_DB  = -3.0
SNR_MAX_DB  = 12.0
N_MIX_VIEWS = 3

RNG = np.random.default_rng(42)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('AUTOSAVE_DIR:', AUTOSAVE_DIR)
print('PERCH_ONNX_PATH:', PERCH_ONNX_PATH)
print('PERCH_LABELS_CSV exists:', PERCH_LABELS_CSV.is_file())
if not PERCH_LABELS_CSV.is_file():
    print('  ⚠ Run: kagglehub.model_download("google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu")')

PROJECT_ROOT: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project
AUTOSAVE_DIR: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/submission_perch_pseudo_labeling/submission_perch_baseline
PERCH_ONNX_PATH: /Users/enricozafiris/.cache/kagglehub/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/versions/2/perch_v2.onnx
PERCH_LABELS_CSV exists: True


## Stage 1 — Load ONNX Perch encoder

In [4]:
# Build the ONNX session
print('Loading Perch ONNX session...', flush=True)
t0 = time.time()
_so = ort.SessionOptions()
_so.intra_op_num_threads = 4
_so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
_perch_session = ort.InferenceSession(
    str(PERCH_ONNX_PATH),
    sess_options=_so,
    providers=['CPUExecutionProvider'],
)
print(f'  Loaded in {time.time()-t0:.2f}s', flush=True)

# Inspect inputs/outputs to learn the names
print('\nModel inputs:')
for inp in _perch_session.get_inputs():
    print(f'  name={inp.name}  shape={inp.shape}  dtype={inp.type}')
print('\nModel outputs:')
for out in _perch_session.get_outputs():
    print(f'  name={out.name}  shape={out.shape}  dtype={out.type}')

_PERCH_INPUT_NAME = _perch_session.get_inputs()[0].name
_PERCH_OUTPUT_NAMES = [o.name for o in _perch_session.get_outputs()]

Loading Perch ONNX session...
  Loaded in 0.41s

Model inputs:
  name=inputs  shape=['batch', 160000]  dtype=tensor(float)

Model outputs:
  name=embedding  shape=['batch', 1536]  dtype=tensor(float)
  name=spatial_embedding  shape=['batch', 16, 4, 1536]  dtype=tensor(float)
  name=spectrogram  shape=['batch', 500, 128]  dtype=tensor(float)
  name=label  shape=['batch', 14795]  dtype=tensor(float)


In [5]:
# Smoke test — ONNX is lightning fast vs TF
print('Smoke test: 1 window through ONNX Perch...', flush=True)
t0 = time.time()
_dummy_in = np.zeros((1, PERCH_SAMPLES), dtype=np.float32)
_outs = _perch_session.run(None, {_PERCH_INPUT_NAME: _dummy_in})
print(f'  First call: {time.time()-t0:.3f}s', flush=True)
for name, arr in zip(_PERCH_OUTPUT_NAMES, _outs):
    print(f'    {name}: shape={arr.shape}  dtype={arr.dtype}')

# Identify which outputs are embedding (1536-d) and logits (~14795-d)
_EMB_IDX = None
_LOGIT_IDX = None
for i, arr in enumerate(_outs):
    if arr.ndim == 2 and arr.shape[-1] == EMB_DIM:
        _EMB_IDX = i
        print(f'  → embedding output is index {i} ({_PERCH_OUTPUT_NAMES[i]})')
    elif arr.ndim == 2 and arr.shape[-1] > 5000:
        _LOGIT_IDX = i
        print(f'  → logits output is index {i} ({_PERCH_OUTPUT_NAMES[i]})  shape={arr.shape}')

if _EMB_IDX is None:
    raise RuntimeError(f'Could not find 1536-d embedding output in {_PERCH_OUTPUT_NAMES}')
if _LOGIT_IDX is None:
    raise RuntimeError(f'Could not find logits output in {_PERCH_OUTPUT_NAMES}')

# Time a few different batch sizes
print('\nBatched throughput test:')
for bs in [1, 8, 16, 32]:
    inp = np.zeros((bs, PERCH_SAMPLES), dtype=np.float32)
    t0 = time.time()
    _ = _perch_session.run(None, {_PERCH_INPUT_NAME: inp})
    dt = time.time() - t0
    print(f'  bs={bs:3d}: {dt:.3f}s total  ({dt/bs*1000:.1f} ms/window)')

Smoke test: 1 window through ONNX Perch...
  First call: 0.080s
    embedding: shape=(1, 1536)  dtype=float32
    spatial_embedding: shape=(1, 16, 4, 1536)  dtype=float32
    spectrogram: shape=(1, 500, 128)  dtype=float32
    label: shape=(1, 14795)  dtype=float32
  → embedding output is index 0 (embedding)
  → logits output is index 3 (label)  shape=(1, 14795)

Batched throughput test:
  bs=  1: 0.075s total  (75.3 ms/window)
  bs=  8: 0.528s total  (66.0 ms/window)
  bs= 16: 1.092s total  (68.3 ms/window)
  bs= 32: 2.215s total  (69.2 ms/window)


In [6]:
# Helper functions
def perch_embed_batch(waveforms_5s):
    """
    Embed a batch of 5-second waveforms (32 kHz) through Perch.
    Args:
        waveforms_5s: list/array of float32, each length PERCH_SAMPLES (160000)
    Returns:
        embeddings: (B, 1536) float32
        logits:     (B, ~14795) float32
    """
    if isinstance(waveforms_5s, list):
        batch = np.stack(waveforms_5s, axis=0).astype(np.float32)
    else:
        batch = np.asarray(waveforms_5s, dtype=np.float32)
    if batch.ndim == 1:
        batch = batch[None, :]
    if batch.shape[1] != PERCH_SAMPLES:
        fixed = np.zeros((batch.shape[0], PERCH_SAMPLES), dtype=np.float32)
        for i, row in enumerate(batch):
            n = min(len(row), PERCH_SAMPLES)
            fixed[i, :n] = row[:n]
        batch = fixed
    outs = _perch_session.run(None, {_PERCH_INPUT_NAME: batch})
    return outs[_EMB_IDX].astype(np.float32), outs[_LOGIT_IDX].astype(np.float32)


def load_focal_clip(path):
    y, _ = librosa.load(str(path), sr=SR_LOAD, mono=True)
    n = PERCH_SAMPLES
    if len(y) >= n:
        s0 = max(0, (len(y) - n) // 2)
        y = y[s0:s0+n]
    else:
        y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD


def soundscape_window_audio(row_id):
    stem, end_s = row_id.rsplit('_', 1)
    end_sec = int(end_s)
    start_sec = max(0, end_sec - int(WINDOW_SEC))
    fp = TRAIN_SOUNDSCAPES_DIR / f'{stem}.ogg'
    y, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True,
                          offset=start_sec, duration=WINDOW_SEC)
    n = PERCH_SAMPLES
    if len(y) > n: y = y[:n]
    elif len(y) < n: y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD


print('Helpers ready.')

Helpers ready.


## Species + labeled soundscape targets

In [7]:
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
species_cols = [c for c in sample_sub.columns if c != 'row_id']
species_to_idx = {s: i for i, s in enumerate(species_cols)}
n_species = len(species_cols)
print('n_species:', n_species)

train_df    = pd.read_csv(TRAIN_CSV)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)

def _tok(v):
    if pd.isna(v) or v == '':
        return set()
    return {t.strip() for t in str(v).split(';') if t.strip()}

def _merge_labels(series):
    o = set()
    for v in series:
        o |= _tok(v)
    return o

lab = pd.read_csv(LABELS_CSV)
grp = (
    lab.groupby(['filename', 'start', 'end'], sort=False)['primary_label']
    .agg(_merge_labels)
    .reset_index()
)
grp['end_sec'] = pd.to_timedelta(grp['end']).dt.total_seconds().astype(int)
grp['row_id']  = grp['filename'].str.replace('.ogg', '', regex=False) + '_' + grp['end_sec'].astype(str)

y_true_rows = []
for _, r in grp.iterrows():
    v = np.zeros(n_species, dtype=np.float32)
    for code in r['primary_label']:
        j = species_to_idx.get(code)
        if j is not None:
            v[j] = 1.0
    y_true_rows.append((r['row_id'], v))

y_true_df = pd.DataFrame(
    [v for _, v in y_true_rows],
    index=[rid for rid, _ in y_true_rows],
    columns=species_cols,
).sort_index()
if not y_true_df.index.is_unique:
    y_true_df = y_true_df.groupby(level=0).max()

labeled_stems = {rid.rsplit('_', 1)[0] for rid in y_true_df.index}
print('Labeled soundscape windows:', len(y_true_df))
print('Labeled soundscape files:  ', len(labeled_stems))

n_species: 234
Labeled soundscape windows: 739
Labeled soundscape files:   66


## Stage 2 — Direct logit mapping + genus proxy

In [8]:
perch_labels = pd.read_csv(PERCH_LABELS_CSV)
print(f'Perch labels columns: {perch_labels.columns.tolist()}')
print(f'Perch vocab size: {len(perch_labels)}')
print(perch_labels.head(3))

sci_col_candidates = [c for c in perch_labels.columns if any(
    k in c.lower() for k in ['sci', 'name', 'inat', 'fsd']
)]
sci_col = sci_col_candidates[0] if sci_col_candidates else perch_labels.columns[0]
print(f"Using column '{sci_col}' for scientific names")

Perch labels columns: ['inat2024_fsd50k']
Perch vocab size: 14795
          inat2024_fsd50k
0      Abavorana luctuosa
1       Abeillia abeillei
2  Abroscopus albogularis
Using column 'inat2024_fsd50k' for scientific names


In [9]:
perch_labels = perch_labels.reset_index().rename(columns={'index': 'perch_idx'})
perch_sci_to_idx = dict(zip(
    perch_labels[sci_col].astype(str).str.strip(),
    perch_labels['perch_idx'].astype(int),
))
NO_LABEL = len(perch_labels)

tax_sci   = taxonomy_df.set_index('primary_label')['scientific_name'].to_dict()
tax_class = taxonomy_df.set_index('primary_label')['class_name'].to_dict()

BC_INDICES = []
for sp in species_cols:
    sci = str(tax_sci.get(sp, '')).strip()
    BC_INDICES.append(perch_sci_to_idx.get(sci, NO_LABEL))
BC_INDICES = np.array(BC_INDICES, dtype=np.int32)

MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)

print(f'Directly mapped: {MAPPED_MASK.sum()}/{n_species}')
print(f'Unmapped:        {(~MAPPED_MASK).sum()}/{n_species}')

Directly mapped: 203/234
Unmapped:        31/234


In [10]:
PROXY_TAXA = {'Aves', 'Amphibia', 'Insecta'}
proxy_map = {}

for sp_idx in UNMAPPED_POS:
    sp  = species_cols[int(sp_idx)]
    cls = tax_class.get(sp, '')
    if cls not in PROXY_TAXA:
        continue
    sci = str(tax_sci.get(sp, '')).strip()
    genus = sci.split()[0] if sci else ''
    if not genus:
        continue
    pat = re.compile(rf'^{re.escape(genus)}\s')
    hits = [pidx for psci, pidx in perch_sci_to_idx.items() if pat.match(psci)]
    if hits:
        proxy_map[int(sp_idx)] = hits

n_unmapped = len(UNMAPPED_POS)
n_proxy = len(proxy_map)
print(f'Unmapped: {n_unmapped} | with genus proxy: {n_proxy} | still no signal: {n_unmapped - n_proxy}')

for idx, bc_idxs in list(proxy_map.items())[:8]:
    sp = species_cols[idx]
    cls = tax_class.get(sp, '?')
    print(f'  {sp:14s} ({cls:10s}) ← {len(bc_idxs)} matches')

def apply_perch_mapping(logits):
    """Convert raw Perch logits (B, vocab) → BirdCLEF species scores (B, 234)."""
    B = logits.shape[0]
    out = np.broadcast_to(
        logits.mean(axis=1, keepdims=True), (B, n_species)
    ).astype(np.float32).copy()
    out[:, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
    for sp_idx, bc_idxs in proxy_map.items():
        out[:, sp_idx] = logits[:, bc_idxs].mean(axis=1)
    return out

Unmapped: 31 | with genus proxy: 3 | still no signal: 28
  1491113        (Amphibia  ) ← 9 matches
  1595929        (Amphibia  ) ← 2 matches
  25073          (Amphibia  ) ← 6 matches


## Stage 3 — Validation cache

In [11]:
FORCE_REBUILD_VAL = False

if not FORCE_REBUILD_VAL and NPZ_VAL.is_file():
    dv = np.load(NPZ_VAL, allow_pickle=True)
    X_val   = dv['X_val'].astype(np.float32)
    S_val   = dv['S_val'].astype(np.float32)
    y_val   = dv['y_val'].astype(np.float32)
    val_ids = list(dv['val_ids'])
    print(f'Loaded val: X={X_val.shape}  S={S_val.shape}  y={y_val.shape}')
else:
    print('Building val cache from labeled soundscape windows...')
    valid_rids, val_audios = [], []
    for rid in list(y_true_df.index):
        stem = rid.rsplit('_', 1)[0]
        if not (TRAIN_SOUNDSCAPES_DIR / f'{stem}.ogg').is_file():
            continue
        y_wav, _ = soundscape_window_audio(rid)
        valid_rids.append(rid)
        val_audios.append(y_wav)
    print(f'  {len(valid_rids)} val windows')

    BATCH = 16
    X_parts, S_parts = [], []
    t0 = time.time()
    for start in tqdm(range(0, len(val_audios), BATCH), desc='Val embed'):
        chunk = val_audios[start:start+BATCH]
        embs, logits = perch_embed_batch(chunk)
        scores = apply_perch_mapping(logits)
        X_parts.append(embs)
        S_parts.append(scores)
    print(f'  Done in {time.time()-t0:.1f}s')

    X_val   = np.concatenate(X_parts, axis=0).astype(np.float32)
    S_val   = np.concatenate(S_parts, axis=0).astype(np.float32)
    rid_va  = np.array(valid_rids, dtype=object)
    y_val   = y_true_df.loc[rid_va].to_numpy(dtype=np.float32)
    val_ids = list(rid_va)
    np.savez_compressed(NPZ_VAL,
                          X_val=X_val, S_val=S_val, y_val=y_val,
                          val_ids=np.array(val_ids, dtype=object))
    print(f'Saved {NPZ_VAL}  X={X_val.shape}  S={S_val.shape}')

Loaded val: X=(739, 1536)  S=(739, 234)  y=(739, 234)


In [12]:
def macro_auc_ge3(y_true, y_score):
    pos = y_true.sum(axis=0)
    keep = pos >= 3
    if not np.any(keep):
        return float('nan'), 0, int(np.sum(pos > 0))
    yt = y_true[:, keep]
    ys = y_score[:, keep]
    usable = []
    for j in range(yt.shape[1]):
        col = yt[:, j]
        if col.min() == 0 and col.max() == 1:
            usable.append(j)
    if not usable:
        return float('nan'), int(np.sum(keep)), int(np.sum(pos > 0))
    usable = np.array(usable, dtype=int)
    auc = roc_auc_score(yt[:, usable], ys[:, usable], average='macro')
    return float(auc), int(np.sum(keep)), int(np.sum(pos > 0))

perch_probs = 1.0 / (1.0 + np.exp(-S_val))
auc_direct, k3, k1 = macro_auc_ge3(y_val, perch_probs)
print('=' * 60)
print(f'BASELINE (direct Perch logits, no training):')
print(f'  AUC>=3 = {auc_direct:.4f}  (k3={k3}, k1={k1})')
print(f'  Compare to BirdNET LB: 0.758')
print('=' * 60)

BASELINE (direct Perch logits, no training):
  AUC>=3 = 0.6346  (k3=64, k1=75)
  Compare to BirdNET LB: 0.758


## Stage 4 — Training cache (focal + soundscape mix)

Set `FOCAL_SUBSET = 3000` for a quick pilot first (~30 min). `None` = all files (~3-6h on M1 with ONNX, much faster than the TF path).

In [13]:
FORCE_REBUILD_TRAIN = False
FOCAL_SUBSET = None    # None = all files; e.g. 3000 for quick pilot

noise_candidates = sorted(TRAIN_SOUNDSCAPES_DIR.glob('*.ogg'))
noise_pool = [p for p in noise_candidates if p.stem not in labeled_stems]
print(f'Noise pool: {len(noise_pool)}')

def random_soundscape_noise_5s():
    fp = Path(noise_pool[int(RNG.integers(0, len(noise_pool)))])
    try:
        dur = librosa.get_duration(path=str(fp))
    except Exception:
        return np.zeros(PERCH_SAMPLES, dtype=np.float32)
    start_max = max(0.0, dur - WINDOW_SEC)
    offset = float(RNG.uniform(0, start_max)) if start_max > 0 else 0.0
    n, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True,
                          offset=offset, duration=WINDOW_SEC)
    nt = PERCH_SAMPLES
    if len(n) > nt: n = n[:nt]
    elif len(n) < nt: n = np.pad(n, (0, nt - len(n)))
    return n.astype(np.float32)

def mix_snr(signal, noise, snr_db):
    s = signal.astype(np.float32)
    n = noise.astype(np.float32)
    ps = np.mean(s ** 2) + 1e-12
    pn = np.mean(n ** 2) + 1e-12
    scale = np.sqrt(ps / (pn * (10 ** (snr_db / 10.0))))
    return np.clip(s + scale * n, -1.0, 1.0)

def build_focal_manifest():
    rows = []
    for _, r in train_df.iterrows():
        lab = str(r['primary_label'])
        if lab not in species_to_idx:
            continue
        ap = TRAIN_AUDIO_DIR / str(r['filename'])
        if not ap.is_file():
            continue
        rows.append({'path': str(ap.resolve()), 'primary_label': lab})
    return pd.DataFrame(rows)

Noise pool: 10592


In [14]:
if not FORCE_REBUILD_TRAIN and NPZ_TRAIN.is_file():
    dt_npz = np.load(NPZ_TRAIN, allow_pickle=True)
    X_train = dt_npz['X_train'].astype(np.float32)
    y_train = dt_npz['y_train'].astype(np.float32)
    print(f'Loaded train: X={X_train.shape}  y={y_train.shape}')
else:
    manifest = build_focal_manifest()
    if FOCAL_SUBSET is not None:
        manifest = manifest.sample(n=min(FOCAL_SUBSET, len(manifest)),
                                    random_state=42).reset_index(drop=True)
    print(f'Focal: {len(manifest)} files x {N_MIX_VIEWS} views')

    BATCH_FILES = 8   # 8 files * 3 views = 24 windows per ONNX call
    Xmx, ymx, paths_meta, var_meta = [], [], [], []
    t_start = time.time()

    for start in tqdm(range(0, len(manifest), BATCH_FILES), desc='Train embed'):
        chunk_rows = manifest.iloc[start:start+BATCH_FILES]
        b_audio, b_labels, b_paths, b_vars = [], [], [], []
        for r in chunk_rows.itertuples(index=False):
            try:
                y_clean, _ = load_focal_clip(Path(r.path))
            except Exception:
                continue
            lbl = np.zeros(n_species, dtype=np.float32)
            lbl[species_to_idx[r.primary_label]] = 1.0
            for v in range(N_MIX_VIEWS):
                if v == 0:
                    y_use = y_clean
                elif noise_pool and RNG.random() < MIX_PROB:
                    noise = random_soundscape_noise_5s()
                    snr = float(RNG.uniform(SNR_MIN_DB, SNR_MAX_DB))
                    y_use = mix_snr(y_clean, noise, snr)
                else:
                    y_use = y_clean
                b_audio.append(y_use)
                b_labels.append(lbl.copy())
                b_paths.append(r.path)
                b_vars.append(np.int16(v))

        if not b_audio:
            continue
        embs, _logits = perch_embed_batch(b_audio)
        Xmx.extend([e for e in embs])
        ymx.extend(b_labels)
        paths_meta.extend(b_paths)
        var_meta.extend(b_vars)

        files_done = start + len(chunk_rows)
        if files_done % 200 == 0 or files_done >= len(manifest):
            elapsed = time.time() - t_start
            rate = files_done / max(elapsed, 1e-3)
            eta_min = (len(manifest) - files_done) / max(rate, 1e-3) / 60
            print(f'  [{files_done}/{len(manifest)}] {rate:.2f} f/s | ETA {eta_min:.1f}m', flush=True)
            gc.collect()

    X_train = np.stack(Xmx).astype(np.float32)
    y_train = np.stack(ymx).astype(np.float32)
    np.savez_compressed(
        NPZ_TRAIN,
        X_train=X_train, y_train=y_train,
        paths=np.array(paths_meta, dtype=object),
        variant=np.array(var_meta),
    )
    print(f'\nSaved {NPZ_TRAIN}  X={X_train.shape}')

Loaded train: X=(106647, 1536)  y=(106647, 234)


## Stage 5 — Train residual MLP head

In [15]:
%pip install -q torch

Note: you may need to restart the kernel to use updated packages.


In [16]:
# pos = y_train.sum(axis=0).astype(np.float64)
# neg = len(y_train) - pos
# pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
# pw_v = tf.constant(pos_weight)[tf.newaxis, :]

# def weighted_multilabel_bce(y_true, y_pred):
#     y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
#     return tf.reduce_mean(
#         pw_v * y_true * (-tf.math.log(y_pred))
#         + (1.0 - y_true) * (-tf.math.log(1.0 - y_pred)),
#         axis=-1,
#     )

# tf.keras.utils.set_random_seed(42)
# inp = tf.keras.layers.Input(shape=(EMB_DIM,))
# x   = tf.keras.layers.BatchNormalization()(inp)


# # Project 1536 → 1024 once so residual stream is 1024-d throughout
# x = tf.keras.layers.Dense(1024, name='input_proj')(x)
# x = tf.keras.layers.LayerNormalization(name='input_proj_ln')(x)


# for i in range(2):
#     h = tf.keras.layers.Dense(1024, name=f'block{i}_dense1')(x)
#     h = tf.keras.layers.LayerNormalization(name=f'block{i}_ln1')(h)
#     h = tf.keras.layers.Activation('gelu', name=f'block{i}_gelu')(h)
#     h = tf.keras.layers.Dropout(0.30, name=f'block{i}_drop')(h)
#     h = tf.keras.layers.Dense(1024, name=f'block{i}_dense2')(h)
#     x = tf.keras.layers.Add(name=f'block{i}_add')([x, h])
#     x = tf.keras.layers.LayerNormalization(name=f'block{i}_ln_out')(x)
# x   = tf.keras.layers.Dense(512, activation='gelu', name='proj')(x)
# x   = tf.keras.layers.Dropout(0.40, name='proj_drop')(x)
# out = tf.keras.layers.Dense(n_species, activation='sigmoid', name='classifier')(x)
# head_perch = tf.keras.Model(inp, out, name='head_perch')
# print(f'MLP head params: {head_perch.count_params():,}')

# head_perch.compile(
#     optimizer=tf.keras.optimizers.Adam(8e-4),
#     loss=weighted_multilabel_bce,
# )

# callbacks = [
#     tf.keras.callbacks.EarlyStopping(
#         patience=7, restore_best_weights=True, monitor='val_loss'),
#     tf.keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
# ]

# perm = np.random.default_rng(42).permutation(len(X_train))
# X_tr_s = X_train[perm]
# y_tr_s = y_train[perm]

# print(f'\nTraining: {len(X_tr_s)} samples, max 80 epochs')
# t0 = time.time()
# hist = head_perch.fit(
#     X_tr_s, y_tr_s,
#     validation_split=0.1,
#     epochs=80,
#     batch_size=256,
#     callbacks=callbacks,
#     verbose=1,
# )
# print(f'Train time: {time.time()-t0:.1f}s | epochs: {len(hist.history["loss"])}')


import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Force CPU on M1 — MPS has its own quirks for this workload
device = torch.device('cpu')
print(f'Using device: {device}', flush=True)

# ── Per-class positive weights ──────────────────────────────────────────
pos = y_train.sum(axis=0).astype(np.float64)
neg = len(y_train) - pos
pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
pw_t = torch.from_numpy(pos_weight).to(device)

# ── Train/val split ─────────────────────────────────────────────────────
split_rng = np.random.default_rng(42)
perm = split_rng.permutation(len(X_train))
n_val = int(0.1 * len(X_train))
val_idx = perm[:n_val]
trn_idx = perm[n_val:]

X_trn = torch.from_numpy(X_train[trn_idx]).float()
y_trn = torch.from_numpy(y_train[trn_idx]).float()
X_vld = torch.from_numpy(X_train[val_idx]).float()
y_vld = torch.from_numpy(y_train[val_idx]).float()
print(f'Train: {len(X_trn)}  Val: {len(X_vld)}', flush=True)

# ── Model: same residual architecture as the TF version ─────────────────
class ResidualHead(nn.Module):
    def __init__(self, emb_dim=1536, n_classes=234, hidden=1024, proj=512,
                 n_blocks=2, dropout_block=0.30, dropout_final=0.40):
        super().__init__()
        self.in_bn   = nn.BatchNorm1d(emb_dim)
        self.in_proj = nn.Linear(emb_dim, hidden)
        self.in_ln   = nn.LayerNorm(hidden)

        self.blocks = nn.ModuleList()
        for _ in range(n_blocks):
            self.blocks.append(nn.ModuleDict({
                'd1': nn.Linear(hidden, hidden),
                'ln1': nn.LayerNorm(hidden),
                'drop': nn.Dropout(dropout_block),
                'd2': nn.Linear(hidden, hidden),
                'ln_out': nn.LayerNorm(hidden),
            }))

        self.proj = nn.Linear(hidden, proj)
        self.proj_drop = nn.Dropout(dropout_final)
        self.classifier = nn.Linear(proj, n_classes)

    def forward(self, x):
        x = self.in_bn(x)
        x = self.in_proj(x)
        x = self.in_ln(x)
        for blk in self.blocks:
            h = blk['d1'](x)
            h = blk['ln1'](h)
            h = F.gelu(h)
            h = blk['drop'](h)
            h = blk['d2'](h)
            x = x + h
            x = blk['ln_out'](x)
        x = F.gelu(self.proj(x))
        x = self.proj_drop(x)
        return self.classifier(x)   # logits

torch.manual_seed(42)
model = ResidualHead(emb_dim=EMB_DIM, n_classes=n_species).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {n_params:,}', flush=True)

# ── Optimizer + loss ────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=8e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-5,
)

def weighted_bce(logits, targets):
    return F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pw_t,
    )

# ── DataLoaders ─────────────────────────────────────────────────────────
BATCH = 256
trn_loader = DataLoader(
    TensorDataset(X_trn, y_trn),
    batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=False,
)
val_loader = DataLoader(
    TensorDataset(X_vld, y_vld),
    batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=False,
)

# ── Training loop ───────────────────────────────────────────────────────
EPOCHS = 80
PATIENCE = 7
best_val = float('inf')
best_state = None
patience_cnt = 0

print(f'\nTraining: max {EPOCHS} epochs', flush=True)
t_total = time.time()

for epoch in range(EPOCHS):
    t_epoch = time.time()

    # Train
    model.train()
    train_losses = []
    for step, (xb, yb) in enumerate(trn_loader):
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = weighted_bce(logits, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        if epoch == 0 and step == 0:
            print(f'  ✓ first step done in {time.time()-t_epoch:.1f}s — training is flowing', flush=True)
        if step > 0 and step % 100 == 0:
            print(f'    epoch {epoch+1} step {step}/{len(trn_loader)}  '
                   f'loss={np.mean(train_losses[-100:]):.4f}', flush=True)

    # Validate
    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            val_losses.append(weighted_bce(logits, yb).item())

    train_loss = float(np.mean(train_losses))
    val_loss   = float(np.mean(val_losses))
    dt = time.time() - t_epoch
    cur_lr = optimizer.param_groups[0]['lr']
    print(f'[Epoch {epoch+1:2d}] loss={train_loss:.4f} val_loss={val_loss:.4f} | lr={cur_lr:.1e} | {dt:.1f}s', flush=True)

    scheduler.step(val_loss)

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1

    if patience_cnt >= PATIENCE:
        print(f'  Early stopping at epoch {epoch+1}', flush=True)
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f'Restored best weights (val_loss={best_val:.4f})', flush=True)

print(f'\nTotal train time: {time.time()-t_total:.1f}s', flush=True)

Using device: cpu
Train: 95983  Val: 10664
Model params: 6,430,442

Training: max 80 epochs
  ✓ first step done in 0.1s — training is flowing
    epoch 1 step 100/375  loss=0.1556
    epoch 1 step 200/375  loss=0.0893
    epoch 1 step 300/375  loss=0.0798
[Epoch  1] loss=0.1031 val_loss=0.0617 | lr=8.0e-04 | 9.1s
    epoch 2 step 100/375  loss=0.0593
    epoch 2 step 200/375  loss=0.0604
    epoch 2 step 300/375  loss=0.0562
[Epoch  2] loss=0.0578 val_loss=0.0505 | lr=8.0e-04 | 10.3s
    epoch 3 step 100/375  loss=0.0447
    epoch 3 step 200/375  loss=0.0456
    epoch 3 step 300/375  loss=0.0455
[Epoch  3] loss=0.0452 val_loss=0.0432 | lr=8.0e-04 | 8.8s
    epoch 4 step 100/375  loss=0.0361
    epoch 4 step 200/375  loss=0.0376
    epoch 4 step 300/375  loss=0.0381
[Epoch  4] loss=0.0375 val_loss=0.0382 | lr=8.0e-04 | 8.8s
    epoch 5 step 100/375  loss=0.0305
    epoch 5 step 200/375  loss=0.0314
    epoch 5 step 300/375  loss=0.0329
[Epoch  5] loss=0.0319 val_loss=0.0352 | lr=8.0e-04

## Stage 6 — Evaluate + blend sweep

In [17]:
model.eval()
with torch.no_grad():
    pred_mlp_logits = model(torch.from_numpy(X_val).float().to(device)).cpu().numpy()
pred_mlp = 1.0 / (1.0 + np.exp(-pred_mlp_logits))
pred_perch = (1 / (1 + np.exp(-S_val))).astype(np.float32)

auc_mlp,   k3_mlp, _ = macro_auc_ge3(y_val, pred_mlp)
auc_perch, k3_p,   _ = macro_auc_ge3(y_val, pred_perch)

print('=' * 60)
print('VALIDATION RESULTS')
print('=' * 60)
print(f'  Direct Perch: AUC>=3 = {auc_perch:.4f}  (k3={k3_p})')
print(f'  MLP head:     AUC>=3 = {auc_mlp:.4f}  (k3={k3_mlp})')

print('\nBlend sweep:')
best_auc, best_w = 0.0, 0.5
for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    blended = w * pred_perch + (1 - w) * pred_mlp
    auc, _, _ = macro_auc_ge3(y_val, blended)
    flag = '  ←' if auc > best_auc else ''
    print(f'  perch={w:.1f}  mlp={1-w:.1f}: AUC>=3 = {auc:.4f}{flag}')
    if auc > best_auc:
        best_auc = auc
        best_w = w

PERCH_WEIGHT = best_w
MLP_WEIGHT = 1 - best_w
print(f'\nBest: perch={PERCH_WEIGHT:.1f}  mlp={MLP_WEIGHT:.1f} → AUC>=3 = {best_auc:.4f}')
print(f'BirdNET LB ref: 0.758 | local val delta: {best_auc - 0.758:+.4f}')

VALIDATION RESULTS
  Direct Perch: AUC>=3 = 0.6346  (k3=64)
  MLP head:     AUC>=3 = 0.5196  (k3=64)

Blend sweep:
  perch=0.0  mlp=1.0: AUC>=3 = 0.5196  ←
  perch=0.1  mlp=0.9: AUC>=3 = 0.6035  ←
  perch=0.2  mlp=0.8: AUC>=3 = 0.6100  ←
  perch=0.3  mlp=0.7: AUC>=3 = 0.6141  ←
  perch=0.4  mlp=0.6: AUC>=3 = 0.6171  ←
  perch=0.5  mlp=0.5: AUC>=3 = 0.6197  ←
  perch=0.6  mlp=0.4: AUC>=3 = 0.6221  ←
  perch=0.7  mlp=0.3: AUC>=3 = 0.6239  ←
  perch=0.8  mlp=0.2: AUC>=3 = 0.6259  ←
  perch=0.9  mlp=0.1: AUC>=3 = 0.6286  ←
  perch=1.0  mlp=0.0: AUC>=3 = 0.6346  ←

Best: perch=1.0  mlp=0.0 → AUC>=3 = 0.6346
BirdNET LB ref: 0.758 | local val delta: -0.1234


## Stage 7 — Save bundle

In [18]:
# Save the PyTorch model state dict
torch.save(model.state_dict(), AUTOSAVE_DIR / 'head_perch_pytorch.pt')

# Save the architecture spec so it can be reloaded on Kaggle
arch_spec = {
    'arch': 'residual_head',
    'emb_dim': int(EMB_DIM),
    'n_classes': int(n_species),
    'hidden': 1024,
    'proj': 512,
    'n_blocks': 2,
    'dropout_block': 0.30,
    'dropout_final': 0.40,
}
(AUTOSAVE_DIR / 'architecture.json').write_text(json.dumps(arch_spec, indent=2))

# Rest of metadata.npz / perch_mapping.npz / metrics.json unchanged

np.savez_compressed(
    AUTOSAVE_DIR / 'metadata.npz',
    species_cols  = np.array(species_cols, dtype=object),
    emb_dim       = np.int32(EMB_DIM),
    sr_load       = np.int32(SR_LOAD),
    perch_sr      = np.int32(PERCH_SR),
    window_sec    = np.float32(WINDOW_SEC),
    n_windows     = np.int32(N_WINDOWS),
    perch_weight  = np.float32(PERCH_WEIGHT),
    mlp_weight    = np.float32(MLP_WEIGHT),
)

proxy_keys = np.array(list(proxy_map.keys()), dtype=np.int32)
proxy_vals = np.array([np.array(v, dtype=np.int32) for v in proxy_map.values()], dtype=object)
np.savez_compressed(
    AUTOSAVE_DIR / 'perch_mapping.npz',
    BC_INDICES    = BC_INDICES,
    MAPPED_POS    = MAPPED_POS,
    MAPPED_BC_IDX = MAPPED_BC_IDX,
    UNMAPPED_POS  = UNMAPPED_POS,
    proxy_keys    = proxy_keys,
    proxy_vals    = proxy_vals,
    NO_LABEL      = np.int32(NO_LABEL),
)

(AUTOSAVE_DIR / 'metrics.json').write_text(json.dumps({
    'auc_direct_perch': float(auc_perch),
    'auc_mlp_head':     float(auc_mlp),
    'auc_best_blend':   float(best_auc),
    'perch_weight':     float(PERCH_WEIGHT),
    'mlp_weight':       float(MLP_WEIGHT),
    'k3_mlp':           int(k3_mlp),
    'k3_perch':         int(k3_p),
    'n_mapped':         int(MAPPED_MASK.sum()),
    'n_proxy':          len(proxy_map),
    'n_unmappable':     int((~MAPPED_MASK).sum()) - len(proxy_map),
    'n_train_samples':  int(len(X_train)),
    'emb_dim':          int(EMB_DIM),
    'birdnet_lb_ref':   0.758,
}, indent=2))

print(f'Bundle saved to: {AUTOSAVE_DIR}')
for p in sorted(AUTOSAVE_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1e6:.2f} MB)')

Bundle saved to: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/submission_perch_pseudo_labeling/submission_perch_baseline
  architecture.json  (0.00 MB)
  head_perch_pytorch.pt  (25.74 MB)
  metadata.npz  (0.00 MB)
  metrics.json  (0.00 MB)
  perch_mapping.npz  (0.00 MB)


## Stage 8 Pseudo Labelling

In [19]:
# # ─────────────────────────────────────────────────────────────────────────
# # Stage 8 — Pseudo-labeling: embed unlabeled soundscapes + filter
# # ─────────────────────────────────────────────────────────────────────────
# import time, gc, json

# # Configuration
# PSEUDO_TOP1_THRESHOLD = 0.50
# PSEUDO_RUNNERUP_MAX   = 0.40
# PSEUDO_BATCH_SIZE     = 16  # Adjust based on memory
# NPZ_PSEUDO = CACHE_DIR / "perch_pseudo_labels.npz"

# # We use the noise_pool created in Stage 4 (unlabeled soundscapes)
# print(f"Starting pseudo-labeling on {len(noise_pool)} files...")

# kept_X, kept_y = [], []
# n_total_windows = 0

# t_start = time.time()

# # Process unlabeled files
# for fi, fp in enumerate(noise_pool):
#     try:
#         # Load full file (approx 4 mins / 240s)
#         y_full, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True)
#         dur = len(y_full) / SR_LOAD
        
#         # Slice into 5s windows
#         windows = []
#         for start_sec in range(0, int(dur) - 4, 5):
#             s0 = int(start_sec * SR_LOAD)
#             s1 = s0 + PERCH_SAMPLES
#             chunk = y_full[s0:s1]
#             if len(chunk) < PERCH_SAMPLES:
#                 chunk = np.pad(chunk, (0, PERCH_SAMPLES - len(chunk)))
#             windows.append(chunk)
        
#         if not windows: continue
            
#         # 1. Get Perch Embeddings (ONNX)
#         # We process the whole file in one batch for speed
#         embs, _ = perch_embed_batch(windows)
        
#         # 2. Get MLP Head Predictions (TF)
#         # We use the head_perch trained in Stage 5
#         probs = head_perch.predict(embs, verbose=0, batch_size=PSEUDO_BATCH_SIZE)
        
#         n_total_windows += len(windows)
        
#         # 3. Apply Filtering Logic
#         for i in range(len(windows)):
#             p = probs[i]
#             top2_idx = np.argsort(-p)[:2]
#             top1_p = p[top2_idx[0]]
#             top2_p = p[top2_idx[1]]
            
#             if top1_p >= PSEUDO_TOP1_THRESHOLD and top2_p < PSEUDO_RUNNERUP_MAX:
#                 lbl = np.zeros(n_species, dtype=np.float32)
#                 lbl[top2_idx[0]] = 1.0
#                 kept_X.append(embs[i].copy())
#                 kept_y.append(lbl)
                
#     except Exception as e:
#         print(f"Error processing {fp.name}: {e}")
#         continue
        
#     if (fi + 1) % 50 == 0:
#         elapsed = time.time() - t_start
#         rate = (fi + 1) / elapsed
#         print(f"  [{fi+1}/{len(noise_pool)}] Kept: {len(kept_X)} | {rate:.2f} files/s")

# print(f"\nPseudo-labeling done. Total kept: {len(kept_X)} windows.")

# # Save pseudo labels so you don't have to re-run this next time
# X_pseudo = np.stack(kept_X).astype(np.float32)
# y_pseudo = np.stack(kept_y).astype(np.float32)
# np.savez_compressed(NPZ_PSEUDO, X_pseudo=X_pseudo, y_pseudo=y_pseudo)

# # ─────────────────────────────────────────────────────────────────────────
# # FINAL STEP: Combined Training
# # ─────────────────────────────────────────────────────────────────────────
# print("\nCombining Focal Training Data + Pseudo-Labels...")

# # Combine with Stage 4 data
# X_combined = np.concatenate([X_train, X_pseudo], axis=0)
# y_combined = np.concatenate([y_train, y_pseudo], axis=0)

# # Shuffle
# perm = np.random.default_rng(42).permutation(len(X_combined))
# X_combined = X_combined[perm]
# y_combined = y_combined[perm]

# print(f"New training set size: {len(X_combined)}")

# # Fine-tune the existing head with a lower learning rate
# head_perch.compile(
#     optimizer=tf.keras.optimizers.Adam(2e-4), # Lower LR for fine-tuning
#     loss=weighted_multilabel_bce,
# )

# print("\nRefining head on augmented data (Pseudo-labels + Focal)...")
# head_perch.fit(
#     X_combined, y_combined,
#     validation_split=0.1,
#     epochs=15, # Fewer epochs for refinement
#     batch_size=256,
#     callbacks=callbacks,
#     verbose=1
# )

# # Save the final improved model
# head_perch.save(AUTOSAVE_DIR / 'head_perch_mlp_final.keras')
# head_perch.save_weights(AUTOSAVE_DIR / 'head_perch_weights_final.weights.h5')
# print(f"\nFinal Refined Bundle saved to: {AUTOSAVE_DIR}")

In [20]:
import torch
import torch.nn as nn
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# ─────────────────────────────────────────────────────────────────────────
# Stage 8 — Fast Pseudo-labeling (M1-optimized, dual threshold)
# ─────────────────────────────────────────────────────────────────────────

# 1. Architecture (unchanged — must match .pt file)
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.d1, self.ln1 = nn.Linear(dim, dim), nn.LayerNorm(dim)
        self.d2, self.ln_out = nn.Linear(dim, dim), nn.LayerNorm(dim)
        self.gelu, self.dropout = nn.GELU(), nn.Dropout(0.30)
    def forward(self, x):
        h = self.gelu(self.ln1(self.d1(x)))
        h = self.dropout(h)
        h = self.d2(h)
        return self.ln_out(x + h)

class ResidualHead(nn.Module):
    def __init__(self, input_dim=1536, hidden_dim=1024, n_classes=234):
        super().__init__()
        self.in_bn = nn.BatchNorm1d(input_dim)
        self.in_proj = nn.Linear(input_dim, hidden_dim)
        self.in_ln = nn.LayerNorm(hidden_dim)
        self.blocks = nn.ModuleList([ResBlock(hidden_dim), ResBlock(hidden_dim)])
        self.proj = nn.Linear(hidden_dim, 512)
        self.classifier = nn.Linear(512, n_classes)
        self.gelu, self.dropout_final = nn.GELU(), nn.Dropout(0.40)
    def forward(self, x):
        x = self.gelu(self.in_ln(self.in_proj(self.in_bn(x))))
        for b in self.blocks:
            x = b(x)
        return torch.sigmoid(self.classifier(self.dropout_final(self.gelu(self.proj(x)))))

# 2. Device — use MPS on M1
device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

pt_path = Path('/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/'
               'APA-Project/notebooks/submission_perch/submission_perch_baseline/'
               'head_perch_pytorch.pt')

model_pt = ResidualHead(input_dim=EMB_DIM, hidden_dim=1024, n_classes=n_species).to(device)
model_pt.load_state_dict(torch.load(pt_path, map_location=device))
model_pt.eval()

# 3. Faster audio loader — soundfile when sample rate matches, librosa as fallback
def load_and_chunk(fp):
    try:
        try:
            y, sr = sf.read(str(fp), dtype='float32', always_2d=False)
            if y.ndim > 1:
                y = y.mean(axis=1)  # mono
            if sr != SR_LOAD:
                y = librosa.resample(y, orig_sr=sr, target_sr=SR_LOAD)
        except Exception:
            y, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True)
        n = len(y) - PERCH_SAMPLES + 1
        return [y[i:i + PERCH_SAMPLES] for i in range(0, n, PERCH_SAMPLES)]
    except Exception as e:
        print(f"  ⚠️  skipped {fp}: {e}")
        return None

# 4. Two threshold profiles — computed in the same pass
THRESHOLDS = {
    'strict':  {'top1': 0.60, 'top2': 0.30},
    'lenient': {'top1': 0.50, 'top2': 0.40},
}

# Per-profile accumulators
kept = {name: {'X': [], 'y': []} for name in THRESHOLDS}

PSEUDO_FILE_BATCH = 16  # bumped up from 8
N_WORKERS = 8           # bumped up from 4

print(f"🚀 Processing {len(noise_pool)} files for pseudo-labeling...")
with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    for i in tqdm(range(0, len(noise_pool), PSEUDO_FILE_BATCH)):
        batch_files = noise_pool[i : i + PSEUDO_FILE_BATCH]
        results = list(executor.map(load_and_chunk, batch_files))
        all_windows = [w for res in results if res for w in res]
        if not all_windows:
            continue

        # Embed once
        embs, _ = perch_embed_batch(all_windows)

        # Score once
        with torch.inference_mode():
            probs = model_pt(torch.from_numpy(embs).to(device)).cpu().numpy()

        # Compute top-2 once, reuse for both filters
        top2 = np.partition(probs, -2, axis=1)[:, -2:]
        top1_vals = top2[:, 1]
        top2_vals = top2[:, 0]
        top_idx_all = np.argmax(probs, axis=1)

        for name, t in THRESHOLDS.items():
            mask = (top1_vals >= t['top1']) & (top2_vals < t['top2'])
            if not np.any(mask):
                continue
            filt_embs = embs[mask]
            filt_idx = top_idx_all[mask]
            labels = np.zeros((len(filt_idx), n_species), dtype=np.float32)
            labels[np.arange(len(filt_idx)), filt_idx] = 1.0
            kept[name]['X'].append(filt_embs)
            kept[name]['y'].append(labels)

# Free PyTorch memory before TF training kicks in
del model_pt
if device.type == 'mps':
    torch.mps.empty_cache()

# ─────────────────────────────────────────────────────────────────────────
# 5. Save both checkpoints immediately
# ─────────────────────────────────────────────────────────────────────────
saved_sets = {}
for name, data in kept.items():
    if not data['X']:
        print(f"⚠️  '{name}' profile: no labels passed thresholds.")
        continue
    X_p = np.concatenate(data['X']).astype(np.float32)
    y_p = np.concatenate(data['y']).astype(np.float32)
    out_path = AUTOSAVE_DIR / f"pseudo_labels_{name}.npz"
    np.savez_compressed(out_path, X_pseudo=X_p, y_pseudo=y_p)
    saved_sets[name] = (X_p, y_p)
    print(f"✅ '{name}': {len(X_p):>6} windows  →  {out_path.name}")

# ─────────────────────────────────────────────────────────────────────────
# 6. Train one model per profile
# ─────────────────────────────────────────────────────────────────────────
for name, (X_p, y_p) in saved_sets.items():
    print(f"\n🎯 Training '{name}' model ({len(X_p)} pseudo windows)")
    X_final = np.concatenate([X_train, X_p], axis=0)
    y_final = np.concatenate([y_train, y_p], axis=0)

    shuffler = np.random.permutation(len(X_final))
    X_final, y_final = X_final[shuffler], y_final[shuffler]

    head_perch.compile(
        optimizer=tf.keras.optimizers.Adam(2e-4),
        loss=weighted_multilabel_bce,
    )
    head_perch.fit(
        X_final, y_final,
        validation_split=0.1,
        epochs=15,
        batch_size=256,
        callbacks=callbacks,
        verbose=1,
    )
    out_model = AUTOSAVE_DIR / f"head_perch_pseudo_{name}.keras"
    head_perch.save(out_model)
    print(f"💾 Saved: {out_model.name}")

print("\n🎊 Done.")

Using device: mps
🚀 Processing 10592 files for pseudo-labeling...


  0%|          | 0/662 [00:00<?, ?it/s]

✅ 'strict':   9001 windows  →  pseudo_labels_strict.npz
✅ 'lenient':  20147 windows  →  pseudo_labels_lenient.npz

🎯 Training 'strict' model (9001 pseudo windows)


NameError: name 'head_perch' is not defined

In [25]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import numpy as np
# import soundfile as sf
# import librosa
# import time
# from pathlib import Path
# from tqdm.auto import tqdm
# from concurrent.futures import ThreadPoolExecutor
# from torch.utils.data import DataLoader, TensorDataset

# # ─────────────────────────────────────────────────────────────────────────
# # Stage 8 — Pseudo-labeling + PyTorch fine-tune (resumable)
# # ─────────────────────────────────────────────────────────────────────────

# # 1. Architecture (must match the .pt file)
# class ResBlock(nn.Module):
#     def __init__(self, dim):
#         super().__init__()
#         self.d1, self.ln1 = nn.Linear(dim, dim), nn.LayerNorm(dim)
#         self.d2, self.ln_out = nn.Linear(dim, dim), nn.LayerNorm(dim)
#         self.gelu, self.dropout = nn.GELU(), nn.Dropout(0.30)
#     def forward(self, x):
#         h = self.gelu(self.ln1(self.d1(x)))
#         h = self.dropout(h)
#         h = self.d2(h)
#         return self.ln_out(x + h)

# class ResidualHead(nn.Module):
#     def __init__(self, input_dim=1536, hidden_dim=1024, n_classes=234):
#         super().__init__()
#         self.in_bn = nn.BatchNorm1d(input_dim)
#         self.in_proj = nn.Linear(input_dim, hidden_dim)
#         self.in_ln = nn.LayerNorm(hidden_dim)
#         self.blocks = nn.ModuleList([ResBlock(hidden_dim), ResBlock(hidden_dim)])
#         self.proj = nn.Linear(hidden_dim, 512)
#         self.classifier = nn.Linear(512, n_classes)
#         self.gelu, self.dropout_final = nn.GELU(), nn.Dropout(0.40)
#     def forward(self, x):
#         x = self.gelu(self.in_ln(self.in_proj(self.in_bn(x))))
#         for b in self.blocks:
#             x = b(x)
#         # Returns SIGMOID probabilities (matches your pseudo-label scoring)
#         return torch.sigmoid(self.classifier(self.dropout_final(self.gelu(self.proj(x)))))

# # Variant that returns LOGITS — needed for BCE-with-logits during fine-tuning
# class ResidualHeadLogits(ResidualHead):
#     def forward(self, x):
#         x = self.gelu(self.in_ln(self.in_proj(self.in_bn(x))))
#         for b in self.blocks:
#             x = b(x)
#         return self.classifier(self.dropout_final(self.gelu(self.proj(x))))

# # 2. Paths & threshold profiles
# THRESHOLDS = {
#     'strict':  {'top1': 0.60, 'top2': 0.30},
#     'lenient': {'top1': 0.50, 'top2': 0.40},
# }
# npz_paths = {name: AUTOSAVE_DIR / f"pseudo_labels_{name}.npz" for name in THRESHOLDS}
# all_exist = all(p.is_file() for p in npz_paths.values())

# # 3. Device for inference & training
# device = torch.device(
#     'mps' if torch.backends.mps.is_available()
#     else 'cuda' if torch.cuda.is_available()
#     else 'cpu'
# )
# print(f"Using device: {device}")

# pt_path = AUTOSAVE_DIR / 'head_perch_pytorch.pt'

# # ─────────────────────────────────────────────────────────────────────────
# # 4. Generate pseudo-labels — SKIP if both npz files already exist
# # ─────────────────────────────────────────────────────────────────────────
# if all_exist:
#     print("✅ Both pseudo-label files already exist — skipping generation.")
#     for name, p in npz_paths.items():
#         print(f"   • {p.name}")
# else:
#     print("🚀 Generating pseudo-labels...")

#     model_pt = ResidualHead(input_dim=EMB_DIM, hidden_dim=1024, n_classes=n_species).to(device)
#     model_pt.load_state_dict(torch.load(pt_path, map_location=device))
#     model_pt.eval()

#     def load_and_chunk(fp):
#         try:
#             try:
#                 y, sr = sf.read(str(fp), dtype='float32', always_2d=False)
#                 if y.ndim > 1:
#                     y = y.mean(axis=1)
#                 if sr != SR_LOAD:
#                     y = librosa.resample(y, orig_sr=sr, target_sr=SR_LOAD)
#             except Exception:
#                 y, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True)
#             n = len(y) - PERCH_SAMPLES + 1
#             return [y[i:i + PERCH_SAMPLES] for i in range(0, n, PERCH_SAMPLES)]
#         except Exception as e:
#             print(f"  ⚠️  skipped {fp}: {e}")
#             return None

#     kept = {name: {'X': [], 'y': []} for name in THRESHOLDS}
#     PSEUDO_FILE_BATCH = 16
#     N_WORKERS = 8

#     print(f"Processing {len(noise_pool)} files...")
#     with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
#         for i in tqdm(range(0, len(noise_pool), PSEUDO_FILE_BATCH)):
#             batch_files = noise_pool[i : i + PSEUDO_FILE_BATCH]
#             results = list(executor.map(load_and_chunk, batch_files))
#             all_windows = [w for res in results if res for w in res]
#             if not all_windows:
#                 continue

#             embs, _ = perch_embed_batch(all_windows)
#             with torch.inference_mode():
#                 probs = model_pt(torch.from_numpy(embs).to(device)).cpu().numpy()

#             top2 = np.partition(probs, -2, axis=1)[:, -2:]
#             top1_vals, top2_vals = top2[:, 1], top2[:, 0]
#             top_idx_all = np.argmax(probs, axis=1)

#             for name, t in THRESHOLDS.items():
#                 mask = (top1_vals >= t['top1']) & (top2_vals < t['top2'])
#                 if not np.any(mask):
#                     continue
#                 filt_embs = embs[mask]
#                 filt_idx = top_idx_all[mask]
#                 labels = np.zeros((len(filt_idx), n_species), dtype=np.float32)
#                 labels[np.arange(len(filt_idx)), filt_idx] = 1.0
#                 kept[name]['X'].append(filt_embs)
#                 kept[name]['y'].append(labels)

#     # Free PyTorch inference model
#     del model_pt
#     if device.type == 'mps':
#         torch.mps.empty_cache()

#     # Save both checkpoints
#     for name, data in kept.items():
#         if not data['X']:
#             print(f"⚠️  '{name}': no labels passed thresholds.")
#             continue
#         X_p = np.concatenate(data['X']).astype(np.float32)
#         y_p = np.concatenate(data['y']).astype(np.float32)
#         np.savez_compressed(npz_paths[name], X_pseudo=X_p, y_pseudo=y_p)
#         print(f"✅ '{name}': {len(X_p):>6} windows  →  {npz_paths[name].name}")

# # ─────────────────────────────────────────────────────────────────────────
# # 5. Load saved pseudo-labels and fine-tune one PyTorch model per profile
# # ─────────────────────────────────────────────────────────────────────────
# saved_sets = {}
# for name, p in npz_paths.items():
#     if p.is_file():
#         d = np.load(p)
#         saved_sets[name] = (d['X_pseudo'].astype(np.float32),
#                             d['y_pseudo'].astype(np.float32))
#         print(f"📂 Loaded {name}: X={saved_sets[name][0].shape}  y={saved_sets[name][1].shape}")

# # Per-class positive weights (computed from original train set, same as cell 25)
# pos = y_train.sum(axis=0).astype(np.float64)
# neg = len(y_train) - pos
# pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
# pw_t = torch.from_numpy(pos_weight).to(device)

# def weighted_bce(logits, targets):
#     return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw_t)

# def fine_tune(name, X_p, y_p, epochs=15, batch_size=256, lr=2e-4, patience=5):
#     print(f"\n🎯 Fine-tuning '{name}' ({len(X_p)} pseudo + {len(X_train)} original)")

#     # Combine + shuffle
#     X_all = np.concatenate([X_train, X_p], axis=0)
#     y_all = np.concatenate([y_train, y_p], axis=0)
#     rng = np.random.default_rng(42)
#     perm = rng.permutation(len(X_all))
#     X_all, y_all = X_all[perm], y_all[perm]

#     # Train/val split
#     n_val = int(0.1 * len(X_all))
#     X_trn = torch.from_numpy(X_all[n_val:]).float()
#     y_trn = torch.from_numpy(y_all[n_val:]).float()
#     X_vld = torch.from_numpy(X_all[:n_val]).float()
#     y_vld = torch.from_numpy(y_all[:n_val]).float()

#     trn_loader = DataLoader(TensorDataset(X_trn, y_trn),
#                             batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(TensorDataset(X_vld, y_vld),
#                             batch_size=batch_size, shuffle=False)

#     # Fresh model loaded from baseline weights (same starting point each run)
#     model_ft = ResidualHeadLogits(input_dim=EMB_DIM, hidden_dim=1024, n_classes=n_species).to(device)
#     model_ft.load_state_dict(torch.load(pt_path, map_location=device))

#     optimizer = torch.optim.Adam(model_ft.parameters(), lr=lr)
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6
#     )

#     best_val = float('inf')
#     best_state = None
#     patience_cnt = 20
#     t0 = time.time()

#     for epoch in range(epochs):
#         model_ft.train()
#         train_losses = []
#         for xb, yb in trn_loader:
#             xb, yb = xb.to(device), yb.to(device)
#             optimizer.zero_grad()
#             loss = weighted_bce(model_ft(xb), yb)
#             loss.backward()
#             optimizer.step()
#             train_losses.append(loss.item())

#         model_ft.eval()
#         val_losses = []
#         with torch.no_grad():
#             for xb, yb in val_loader:
#                 xb, yb = xb.to(device), yb.to(device)
#                 val_losses.append(weighted_bce(model_ft(xb), yb).item())

#         tl, vl = float(np.mean(train_losses)), float(np.mean(val_losses))
#         cur_lr = optimizer.param_groups[0]['lr']
#         print(f"[Epoch {epoch+1:2d}] loss={tl:.4f} val={vl:.4f} | lr={cur_lr:.1e}", flush=True)
#         scheduler.step(vl)

#         if vl < best_val - 1e-4:
#             best_val = vl
#             best_state = {k: v.clone() for k, v in model_ft.state_dict().items()}
#             patience_cnt = 0
#         else:
#             patience_cnt += 1
#         if patience_cnt >= patience:
#             print(f"  Early stopping at epoch {epoch+1}")
#             break

#     if best_state is not None:
#         model_ft.load_state_dict(best_state)
#     print(f"  Best val_loss={best_val:.4f}  |  {time.time()-t0:.1f}s")

#     out_path = AUTOSAVE_DIR / f"head_perch_pseudo_{name}.pt"
#     torch.save(model_ft.state_dict(), out_path)
#     print(f"💾 Saved: {out_path.name}")

#     del model_ft
#     if device.type == 'mps':
#         torch.mps.empty_cache()

# for name, (X_p, y_p) in saved_sets.items():
#     fine_tune(name, X_p, y_p)

# print("\n🎊 Done.")



import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import soundfile as sf
import librosa
import time
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import DataLoader, TensorDataset

# ─────────────────────────────────────────────────────────────────────────
# Stage 8 — Pseudo-labeling + PyTorch fine-tune (resumable)
# ─────────────────────────────────────────────────────────────────────────

# 1. Architecture — EXACTLY matches cell 25 (the one that trained head_perch_pytorch.pt)
class ResidualHeadLogits(nn.Module):
    """Returns logits. Used for training (BCEWithLogitsLoss) AND for pseudo-label
    scoring (just apply sigmoid manually to the output)."""
    def __init__(self, emb_dim=1536, n_classes=234, hidden=1024, proj=512,
                 n_blocks=2, dropout_block=0.30, dropout_final=0.40):
        super().__init__()
        self.in_bn   = nn.BatchNorm1d(emb_dim)
        self.in_proj = nn.Linear(emb_dim, hidden)
        self.in_ln   = nn.LayerNorm(hidden)

        self.blocks = nn.ModuleList()
        for _ in range(n_blocks):
            self.blocks.append(nn.ModuleDict({
                'd1':     nn.Linear(hidden, hidden),
                'ln1':    nn.LayerNorm(hidden),
                'drop':   nn.Dropout(dropout_block),
                'd2':     nn.Linear(hidden, hidden),
                'ln_out': nn.LayerNorm(hidden),
            }))

        self.proj       = nn.Linear(hidden, proj)
        self.proj_drop  = nn.Dropout(dropout_final)
        self.classifier = nn.Linear(proj, n_classes)

    def forward(self, x):
        x = self.in_bn(x)
        x = self.in_proj(x)
        x = self.in_ln(x)                  # NO gelu here
        for blk in self.blocks:
            h = blk['d1'](x)
            h = blk['ln1'](h)
            h = F.gelu(h)
            h = blk['drop'](h)
            h = blk['d2'](h)
            x = x + h
            x = blk['ln_out'](x)
        x = F.gelu(self.proj(x))
        x = self.proj_drop(x)
        return self.classifier(x)          # logits


class ResidualHead(ResidualHeadLogits):
    """Same network, but returns sigmoid probabilities. Used for pseudo-label scoring."""
    def forward(self, x):
        return torch.sigmoid(super().forward(x))


# 2. Paths & threshold profiles
THRESHOLDS = {
    'strict':  {'top1': 0.60, 'top2': 0.30},
    'lenient': {'top1': 0.50, 'top2': 0.40},
}
npz_paths = {name: AUTOSAVE_DIR / f"pseudo_labels_{name}.npz" for name in THRESHOLDS}
all_exist = all(p.is_file() for p in npz_paths.values())

# 3. Device
device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

pt_path = AUTOSAVE_DIR / 'head_perch_pytorch.pt'

# ─────────────────────────────────────────────────────────────────────────
# 4. Generate pseudo-labels — SKIP if both npz files already exist
# ─────────────────────────────────────────────────────────────────────────
if all_exist:
    print("✅ Both pseudo-label files already exist — skipping generation.")
    for name, p in npz_paths.items():
        print(f"   • {p.name}")
else:
    print("🚀 Generating pseudo-labels...")

    model_pt = ResidualHead(emb_dim=EMB_DIM, n_classes=n_species).to(device)
    model_pt.load_state_dict(torch.load(pt_path, map_location=device))
    model_pt.eval()

    def load_and_chunk(fp):
        try:
            try:
                y, sr = sf.read(str(fp), dtype='float32', always_2d=False)
                if y.ndim > 1:
                    y = y.mean(axis=1)
                if sr != SR_LOAD:
                    y = librosa.resample(y, orig_sr=sr, target_sr=SR_LOAD)
            except Exception:
                y, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True)
            n = len(y) - PERCH_SAMPLES + 1
            return [y[i:i + PERCH_SAMPLES] for i in range(0, n, PERCH_SAMPLES)]
        except Exception as e:
            print(f"  ⚠️  skipped {fp}: {e}")
            return None

    kept = {name: {'X': [], 'y': []} for name in THRESHOLDS}
    PSEUDO_FILE_BATCH = 16
    N_WORKERS = 8

    print(f"Processing {len(noise_pool)} files...")
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        for i in tqdm(range(0, len(noise_pool), PSEUDO_FILE_BATCH)):
            batch_files = noise_pool[i : i + PSEUDO_FILE_BATCH]
            results = list(executor.map(load_and_chunk, batch_files))
            all_windows = [w for res in results if res for w in res]
            if not all_windows:
                continue

            embs, _ = perch_embed_batch(all_windows)
            with torch.inference_mode():
                probs = model_pt(torch.from_numpy(embs).to(device)).cpu().numpy()

            top2 = np.partition(probs, -2, axis=1)[:, -2:]
            top1_vals, top2_vals = top2[:, 1], top2[:, 0]
            top_idx_all = np.argmax(probs, axis=1)

            for name, t in THRESHOLDS.items():
                mask = (top1_vals >= t['top1']) & (top2_vals < t['top2'])
                if not np.any(mask):
                    continue
                filt_embs = embs[mask]
                filt_idx = top_idx_all[mask]
                labels = np.zeros((len(filt_idx), n_species), dtype=np.float32)
                labels[np.arange(len(filt_idx)), filt_idx] = 1.0
                kept[name]['X'].append(filt_embs)
                kept[name]['y'].append(labels)

    del model_pt
    if device.type == 'mps':
        torch.mps.empty_cache()

    for name, data in kept.items():
        if not data['X']:
            print(f"⚠️  '{name}': no labels passed thresholds.")
            continue
        X_p = np.concatenate(data['X']).astype(np.float32)
        y_p = np.concatenate(data['y']).astype(np.float32)
        np.savez_compressed(npz_paths[name], X_pseudo=X_p, y_pseudo=y_p)
        print(f"✅ '{name}': {len(X_p):>6} windows  →  {npz_paths[name].name}")

# ─────────────────────────────────────────────────────────────────────────
# 5. Load saved pseudo-labels and fine-tune one PyTorch model per profile
# ─────────────────────────────────────────────────────────────────────────
saved_sets = {}
for name, p in npz_paths.items():
    if p.is_file():
        d = np.load(p)
        saved_sets[name] = (d['X_pseudo'].astype(np.float32),
                            d['y_pseudo'].astype(np.float32))
        print(f"📂 Loaded {name}: X={saved_sets[name][0].shape}  y={saved_sets[name][1].shape}")

# Per-class positive weights (computed from original train set)
pos = y_train.sum(axis=0).astype(np.float64)
neg = len(y_train) - pos
pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
pw_t = torch.from_numpy(pos_weight).to(device)

def weighted_bce(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw_t)

def fine_tune(name, X_p, y_p, epochs=15, batch_size=256, lr=2e-4, patience=5):
    print(f"\n🎯 Fine-tuning '{name}' ({len(X_p)} pseudo + {len(X_train)} original)")

    X_all = np.concatenate([X_train, X_p], axis=0)
    y_all = np.concatenate([y_train, y_p], axis=0)
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(X_all))
    X_all, y_all = X_all[perm], y_all[perm]

    n_val = int(0.1 * len(X_all))
    X_trn = torch.from_numpy(X_all[n_val:]).float()
    y_trn = torch.from_numpy(y_all[n_val:]).float()
    X_vld = torch.from_numpy(X_all[:n_val]).float()
    y_vld = torch.from_numpy(y_all[:n_val]).float()

    trn_loader = DataLoader(TensorDataset(X_trn, y_trn),
                            batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_vld, y_vld),
                            batch_size=batch_size, shuffle=False)

    model_ft = ResidualHeadLogits(emb_dim=EMB_DIM, n_classes=n_species).to(device)
    model_ft.load_state_dict(torch.load(pt_path, map_location=device))

    optimizer = torch.optim.Adam(model_ft.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6
    )

    best_val = float('inf')
    best_state = None
    patience_cnt = 0   # was 20 in your version — that was a bug, should start at 0
    t0 = time.time()

    for epoch in range(epochs):
        model_ft.train()
        train_losses = []
        for xb, yb in trn_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = weighted_bce(model_ft(xb), yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model_ft.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(weighted_bce(model_ft(xb), yb).item())

        tl, vl = float(np.mean(train_losses)), float(np.mean(val_losses))
        cur_lr = optimizer.param_groups[0]['lr']
        print(f"[Epoch {epoch+1:2d}] loss={tl:.4f} val={vl:.4f} | lr={cur_lr:.1e}", flush=True)
        scheduler.step(vl)

        if vl < best_val - 1e-4:
            best_val = vl
            best_state = {k: v.clone() for k, v in model_ft.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model_ft.load_state_dict(best_state)
    print(f"  Best val_loss={best_val:.4f}  |  {time.time()-t0:.1f}s")

    out_path = AUTOSAVE_DIR / f"head_perch_pseudo_{name}.pt"
    torch.save(model_ft.state_dict(), out_path)
    print(f"💾 Saved: {out_path.name}")

    del model_ft
    if device.type == 'mps':
        torch.mps.empty_cache()

for name, (X_p, y_p) in saved_sets.items():
    fine_tune(name, X_p, y_p)

print("\n🎊 Done.")

Using device: mps
✅ Both pseudo-label files already exist — skipping generation.
   • pseudo_labels_strict.npz
   • pseudo_labels_lenient.npz
📂 Loaded strict: X=(9001, 1536)  y=(9001, 234)
📂 Loaded lenient: X=(20147, 1536)  y=(20147, 234)

🎯 Fine-tuning 'strict' (9001 pseudo + 106647 original)
[Epoch  1] loss=0.0168 val=0.0129 | lr=2.0e-04
[Epoch  2] loss=0.0122 val=0.0127 | lr=2.0e-04
[Epoch  3] loss=0.0102 val=0.0131 | lr=2.0e-04
[Epoch  4] loss=0.0090 val=0.0130 | lr=2.0e-04
[Epoch  5] loss=0.0082 val=0.0135 | lr=2.0e-04
[Epoch  6] loss=0.0072 val=0.0137 | lr=1.0e-04
[Epoch  7] loss=0.0064 val=0.0139 | lr=1.0e-04
  Early stopping at epoch 7
  Best val_loss=0.0127  |  31.2s
💾 Saved: head_perch_pseudo_strict.pt

🎯 Fine-tuning 'lenient' (20147 pseudo + 106647 original)
[Epoch  1] loss=0.0248 val=0.0174 | lr=2.0e-04
[Epoch  2] loss=0.0191 val=0.0174 | lr=2.0e-04
[Epoch  3] loss=0.0164 val=0.0171 | lr=2.0e-04
[Epoch  4] loss=0.0148 val=0.0180 | lr=2.0e-04
[Epoch  5] loss=0.0134 val=0.018

In [29]:
# # ─────────────────────────────────────────────────────────────────────────
# # Stage 9 — Validation AUC for baseline + both pseudo-refined models
# # Reports two metrics:
# #   • macro_auc_ge3  — the local stability-friendly version (your existing one)
# #   • kaggle_auc     — exact replica of competition scorer (sums > 0 filter)
# # ─────────────────────────────────────────────────────────────────────────

# import numpy as np
# import pandas as pd
# import torch
# import pandas.api.types
# from sklearn.metrics import roc_auc_score


# # Kaggle-style scorer (lifted from competition metric file)
# class _ParticipantVisibleError(Exception):
#     pass

# def kaggle_macro_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
#     """Macro ROC-AUC, ignoring classes with no true positives — matches the Kaggle scorer."""
#     cols = [f"c{i}" for i in range(y_true.shape[1])]
#     sol = pd.DataFrame(y_true, columns=cols)
#     sub = pd.DataFrame(y_pred, columns=cols)

#     if not pandas.api.types.is_numeric_dtype(sub.values):
#         raise _ParticipantVisibleError("Non-numeric submission values.")

#     sums = sol.sum(axis=0)
#     scored = list(sums[sums > 0].index.values)
#     if not scored:
#         raise _ParticipantVisibleError("No classes with positive labels.")

#     return roc_auc_score(sol[scored].values, sub[scored].values, average="macro")


# def predict_probs(state_dict_path, X):
#     """Load a PyTorch head and return sigmoid probabilities on X."""
#     m = ResidualHeadLogits(input_dim=EMB_DIM, hidden_dim=1024, n_classes=n_species).to(device)
#     m.load_state_dict(torch.load(state_dict_path, map_location=device))
#     m.eval()
#     with torch.inference_mode():
#         logits = m(torch.from_numpy(X).float().to(device)).cpu().numpy()
#     del m
#     if device.type == 'mps':
#         torch.mps.empty_cache()
#     return 1.0 / (1.0 + np.exp(-logits))


# # 1. Direct Perch baseline (no head training)
# pred_perch = (1.0 / (1.0 + np.exp(-S_val))).astype(np.float32)

# # 2. Collect predictions from each available checkpoint
# candidates = {
#     'baseline (no pseudo)': AUTOSAVE_DIR / 'head_perch_pytorch.pt',
#     'pseudo-strict':        AUTOSAVE_DIR / 'head_perch_pseudo_strict.pt',
#     'pseudo-lenient':       AUTOSAVE_DIR / 'head_perch_pseudo_lenient.pt',
# }

# results = {'direct Perch': pred_perch}
# for name, path in candidates.items():
#     if path.is_file():
#         results[name] = predict_probs(path, X_val)
#         print(f"✓ Loaded predictions from {path.name}")
#     else:
#         print(f"✗ Skipping {name}: {path.name} not found")

# # 3. Score every prediction set with both metrics
# print("\n" + "=" * 72)
# print(f"{'Model':<26} {'macro_auc_ge3':>15} {'kaggle_auc':>15}  {'k3':>4}")
# print("=" * 72)

# scored = {}
# for name, pred in results.items():
#     auc_ge3, k3, _ = macro_auc_ge3(y_val, pred)
#     try:
#         auc_kg = kaggle_macro_auc(y_val, pred)
#     except _ParticipantVisibleError as e:
#         auc_kg = float('nan')
#         print(f"  ⚠️  Kaggle metric failed for {name}: {e}")
#     scored[name] = {'macro_auc_ge3': auc_ge3, 'kaggle_auc': auc_kg, 'k3': k3, 'pred': pred}
#     print(f"{name:<26} {auc_ge3:>15.4f} {auc_kg:>15.4f}  {k3:>4d}")
# print("=" * 72)
# print(f"BirdNET LB ref: 0.758")

# # 4. Quick blend sweep on the best individual head against direct Perch
# head_keys = [k for k in scored if k != 'direct Perch']
# if head_keys:
#     best_head = max(head_keys, key=lambda k: scored[k]['kaggle_auc'])
#     print(f"\nBlend sweep: direct Perch ⨯ {best_head}")
#     print(f"{'perch_w':>8} {'mlp_w':>8} {'macro_auc_ge3':>15} {'kaggle_auc':>15}")
#     pred_head = scored[best_head]['pred']
#     best_auc, best_w = -1.0, 0.5
#     for w in np.arange(0.0, 1.01, 0.1):
#         blend = w * pred_perch + (1 - w) * pred_head
#         a_ge3, _, _ = macro_auc_ge3(y_val, blend)
#         a_kg = kaggle_macro_auc(y_val, blend)
#         flag = '  ←' if a_kg > best_auc else ''
#         print(f"{w:>8.1f} {1-w:>8.1f} {a_ge3:>15.4f} {a_kg:>15.4f}{flag}")
#         if a_kg > best_auc:
#             best_auc, best_w = a_kg, w
#     print(f"\nBest blend (by kaggle_auc): perch={best_w:.1f}  mlp={1-best_w:.1f}  →  {best_auc:.4f}")


# ─────────────────────────────────────────────────────────────────────────
# Stage 9 — Validation AUC for baseline + both pseudo-refined models
# Reports two metrics:
#   • macro_auc_ge3  — the local stability-friendly version (your existing one)
#   • kaggle_auc     — exact replica of competition scorer (sums > 0 filter)
# ─────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import torch
import pandas.api.types
from sklearn.metrics import roc_auc_score


# Kaggle-style scorer (lifted from competition metric file)
class _ParticipantVisibleError(Exception):
    pass

def kaggle_macro_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Macro ROC-AUC, ignoring classes with no true positives — matches the Kaggle scorer."""
    cols = [f"c{i}" for i in range(y_true.shape[1])]
    sol = pd.DataFrame(y_true, columns=cols)
    sub = pd.DataFrame(y_pred, columns=cols)

    if not pandas.api.types.is_numeric_dtype(sub.values):
        raise _ParticipantVisibleError("Non-numeric submission values.")

    sums = sol.sum(axis=0)
    scored = list(sums[sums > 0].index.values)
    if not scored:
        raise _ParticipantVisibleError("No classes with positive labels.")

    return roc_auc_score(sol[scored].values, sub[scored].values, average="macro")


def predict_probs(state_dict_path, X):
    """Load a PyTorch head and return sigmoid probabilities on X."""
    m = ResidualHeadLogits(emb_dim=EMB_DIM, n_classes=n_species).to(device)
    m.load_state_dict(torch.load(state_dict_path, map_location=device))
    m.eval()
    with torch.inference_mode():
        logits = m(torch.from_numpy(X).float().to(device)).cpu().numpy()
    del m
    if device.type == 'mps':
        torch.mps.empty_cache()
    return 1.0 / (1.0 + np.exp(-logits))


# 1. Direct Perch baseline (no head training)
pred_perch = (1.0 / (1.0 + np.exp(-S_val))).astype(np.float32)

# 2. Collect predictions from each available checkpoint
candidates = {
    'baseline (no pseudo)': AUTOSAVE_DIR / 'head_perch_pytorch.pt',
    'pseudo-strict':        AUTOSAVE_DIR / 'head_perch_pseudo_strict.pt',
    'pseudo-lenient':       AUTOSAVE_DIR / 'head_perch_pseudo_lenient.pt',
}

results = {'direct Perch': pred_perch}
for name, path in candidates.items():
    if path.is_file():
        results[name] = predict_probs(path, X_val)
        print(f"✓ Loaded predictions from {path.name}")
    else:
        print(f"✗ Skipping {name}: {path.name} not found")

# 3. Score every prediction set with both metrics
print("\n" + "=" * 72)
print(f"{'Model':<26} {'macro_auc_ge3':>15} {'kaggle_auc':>15}  {'k3':>4}")
print("=" * 72)

scored = {}
for name, pred in results.items():
    auc_ge3, k3, _ = macro_auc_ge3(y_val, pred)
    try:
        auc_kg = kaggle_macro_auc(y_val, pred)
    except _ParticipantVisibleError as e:
        auc_kg = float('nan')
        print(f"  ⚠️  Kaggle metric failed for {name}: {e}")
    scored[name] = {'macro_auc_ge3': auc_ge3, 'kaggle_auc': auc_kg, 'k3': k3, 'pred': pred}
    print(f"{name:<26} {auc_ge3:>15.4f} {auc_kg:>15.4f}  {k3:>4d}")
print("=" * 72)
print(f"BirdNET LB ref: 0.758")

# 4. Quick blend sweep on the best individual head against direct Perch
head_keys = [k for k in scored if k != 'direct Perch']
if head_keys:
    best_head = max(head_keys, key=lambda k: scored[k]['kaggle_auc'])
    print(f"\nBlend sweep: direct Perch ⨯ {best_head}")
    print(f"{'perch_w':>8} {'mlp_w':>8} {'macro_auc_ge3':>15} {'kaggle_auc':>15}")
    pred_head = scored[best_head]['pred']
    best_auc, best_w = -1.0, 0.5
    for w in np.arange(0.0, 1.01, 0.1):
        blend = w * pred_perch + (1 - w) * pred_head
        a_ge3, _, _ = macro_auc_ge3(y_val, blend)
        a_kg = kaggle_macro_auc(y_val, blend)
        flag = '  ←' if a_kg > best_auc else ''
        print(f"{w:>8.1f} {1-w:>8.1f} {a_ge3:>15.4f} {a_kg:>15.4f}{flag}")
        if a_kg > best_auc:
            best_auc, best_w = a_kg, w
    print(f"\nBest blend (by kaggle_auc): perch={best_w:.1f}  mlp={1-best_w:.1f}  →  {best_auc:.4f}")

✓ Loaded predictions from head_perch_pytorch.pt
✓ Loaded predictions from head_perch_pseudo_strict.pt
✓ Loaded predictions from head_perch_pseudo_lenient.pt

Model                        macro_auc_ge3      kaggle_auc    k3
direct Perch                        0.6346          0.6761    64
baseline (no pseudo)                0.5196          0.5350    64
pseudo-strict                       0.5260          0.5401    64
pseudo-lenient                      0.5466          0.5681    64
BirdNET LB ref: 0.758

Blend sweep: direct Perch ⨯ pseudo-lenient
 perch_w    mlp_w   macro_auc_ge3      kaggle_auc
     0.0      1.0          0.5466          0.5681  ←
     0.1      0.9          0.6081          0.6460  ←
     0.2      0.8          0.6130          0.6513  ←
     0.3      0.7          0.6159          0.6548  ←
     0.4      0.6          0.6182          0.6577  ←
     0.5      0.5          0.6203          0.6603  ←
     0.6      0.4          0.6222          0.6628  ←
     0.7      0.3          0.6

## Ensemble Head

In [30]:
# # ─────────────────────────────────────────────────────────────────────────
# # Stage 10 — Diverse head ensemble
# # Train two extra small heads (linear probe + shallow MLP) on the lenient
# # pseudo-augmented set, then ensemble with the existing residual head.
# # ─────────────────────────────────────────────────────────────────────────

# import time
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, TensorDataset

# # ── 1. Define two new lightweight heads ─────────────────────────────────
# class LinearHead(nn.Module):
#     """BN → Linear. Minimal, robust, hard to overfit."""
#     def __init__(self, emb_dim=1536, n_classes=234):
#         super().__init__()
#         self.bn = nn.BatchNorm1d(emb_dim)
#         self.fc = nn.Linear(emb_dim, n_classes)
#     def forward(self, x):
#         return self.fc(self.bn(x))   # logits


# class ShallowMLPHead(nn.Module):
#     """BN → 512 → GELU → Dropout → 234. One hidden layer, no residuals."""
#     def __init__(self, emb_dim=1536, n_classes=234, hidden=512, dropout=0.30):
#         super().__init__()
#         self.bn = nn.BatchNorm1d(emb_dim)
#         self.fc1 = nn.Linear(emb_dim, hidden)
#         self.ln = nn.LayerNorm(hidden)
#         self.drop = nn.Dropout(dropout)
#         self.fc2 = nn.Linear(hidden, n_classes)
#     def forward(self, x):
#         x = self.bn(x)
#         x = F.gelu(self.ln(self.fc1(x)))
#         x = self.drop(x)
#         return self.fc2(x)   # logits


# # ── 2. Pick which pseudo-set to train on ────────────────────────────────
# # Lenient gave the best AUC last round — go with it.
# PROFILE = 'lenient'
# npz = np.load(AUTOSAVE_DIR / f'pseudo_labels_{PROFILE}.npz')
# X_p, y_p = npz['X_pseudo'].astype(np.float32), npz['y_pseudo'].astype(np.float32)

# X_all = np.concatenate([X_train, X_p], axis=0)
# y_all = np.concatenate([y_train, y_p], axis=0)
# rng = np.random.default_rng(42)
# perm = rng.permutation(len(X_all))
# X_all, y_all = X_all[perm], y_all[perm]

# n_val = int(0.1 * len(X_all))
# X_trn = torch.from_numpy(X_all[n_val:]).float()
# y_trn = torch.from_numpy(y_all[n_val:]).float()
# X_vld = torch.from_numpy(X_all[:n_val]).float()
# y_vld = torch.from_numpy(y_all[:n_val]).float()

# # Reuse the per-class pos_weight already computed in Stage 5
# def weighted_bce(logits, targets):
#     return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw_t)


# # ── 3. Generic train loop ───────────────────────────────────────────────
# def train_head(model_cls, name, epochs=30, lr=1e-3, batch_size=256, patience=5):
#     print(f"\n🎯 Training {name}")
#     model = model_cls().to(device)
#     n_params = sum(p.numel() for p in model.parameters())
#     print(f"   params: {n_params:,}")

#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6
#     )
#     trn_loader = DataLoader(TensorDataset(X_trn, y_trn), batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(TensorDataset(X_vld, y_vld), batch_size=batch_size, shuffle=False)

#     best_val, best_state, patience_cnt = float('inf'), None, 0
#     t0 = time.time()
#     for ep in range(epochs):
#         model.train()
#         tl_losses = []
#         for xb, yb in trn_loader:
#             xb, yb = xb.to(device), yb.to(device)
#             optimizer.zero_grad()
#             loss = weighted_bce(model(xb), yb)
#             loss.backward()
#             optimizer.step()
#             tl_losses.append(loss.item())

#         model.eval()
#         vl_losses = []
#         with torch.no_grad():
#             for xb, yb in val_loader:
#                 xb, yb = xb.to(device), yb.to(device)
#                 vl_losses.append(weighted_bce(model(xb), yb).item())

#         tl, vl = float(np.mean(tl_losses)), float(np.mean(vl_losses))
#         print(f"   [Epoch {ep+1:2d}] loss={tl:.4f}  val={vl:.4f}  lr={optimizer.param_groups[0]['lr']:.1e}", flush=True)
#         scheduler.step(vl)

#         if vl < best_val - 1e-4:
#             best_val, best_state, patience_cnt = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
#         else:
#             patience_cnt += 1
#         if patience_cnt >= patience:
#             print(f"   Early stop at epoch {ep+1}")
#             break

#     if best_state is not None:
#         model.load_state_dict(best_state)
#     print(f"   Best val={best_val:.4f}  ({time.time()-t0:.1f}s)")
#     out = AUTOSAVE_DIR / f'head_{name}_{PROFILE}.pt'
#     torch.save(model.state_dict(), out)
#     print(f"   💾 {out.name}")
#     return model


# linear_model  = train_head(LinearHead,     'linear',  epochs=30, lr=1e-3)
# shallow_model = train_head(ShallowMLPHead, 'shallow', epochs=30, lr=8e-4)


# # ── 4. Get predictions from all four sources ────────────────────────────
# def predict_logits(model, X):
#     model.eval()
#     with torch.inference_mode():
#         return model(torch.from_numpy(X).float().to(device)).cpu().numpy()

# def sig(x):
#     return 1.0 / (1.0 + np.exp(-x))

# # Existing residual head (lenient version)
# residual_model = ResidualHeadLogits(input_dim=EMB_DIM, hidden_dim=1024, n_classes=n_species).to(device)
# residual_model.load_state_dict(torch.load(AUTOSAVE_DIR / f'head_perch_pseudo_{PROFILE}.pt', map_location=device))

# pred_lin   = sig(predict_logits(linear_model,   X_val))
# pred_shal  = sig(predict_logits(shallow_model,  X_val))
# pred_res   = sig(predict_logits(residual_model, X_val))
# pred_perch = (1.0 / (1.0 + np.exp(-S_val))).astype(np.float32)


# # ── 5. Score individuals + ensemble combinations ────────────────────────
# def score(name, pred):
#     a_ge3, k3, _ = macro_auc_ge3(y_val, pred)
#     a_kg = kaggle_macro_auc(y_val, pred)
#     print(f"  {name:<32} ge3={a_ge3:.4f}  kaggle={a_kg:.4f}  k3={k3}")
#     return a_kg

# print("\n" + "=" * 72)
# print("INDIVIDUAL HEADS")
# print("=" * 72)
# score('direct Perch',           pred_perch)
# score('linear',                 pred_lin)
# score('shallow MLP',            pred_shal)
# score('residual (lenient)',     pred_res)

# print("\n" + "=" * 72)
# print("ENSEMBLES (equal-weight average of MLP heads)")
# print("=" * 72)
# ens_lin_shal      = (pred_lin + pred_shal) / 2
# ens_lin_res       = (pred_lin + pred_res) / 2
# ens_shal_res      = (pred_shal + pred_res) / 2
# ens_all_three     = (pred_lin + pred_shal + pred_res) / 3
# score('lin + shal',             ens_lin_shal)
# score('lin + res',              ens_lin_res)
# score('shal + res',             ens_shal_res)
# score('lin + shal + res',       ens_all_three)


# # ── 6. Best-MLP-ensemble × direct Perch sweep ───────────────────────────
# candidates = {
#     'lin + shal':         ens_lin_shal,
#     'lin + res':          ens_lin_res,
#     'shal + res':         ens_shal_res,
#     'lin + shal + res':   ens_all_three,
# }
# best_ens_name = max(candidates, key=lambda k: kaggle_macro_auc(y_val, candidates[k]))
# best_ens_pred = candidates[best_ens_name]
# print(f"\nBest MLP ensemble: {best_ens_name}")

# print(f"\nBlend sweep: direct Perch ⨯ ({best_ens_name})")
# print(f"{'perch_w':>8} {'mlp_w':>8} {'kaggle_auc':>12}")
# best_auc, best_w = -1, 0.5
# for w in np.arange(0.0, 1.01, 0.05):
#     blend = w * pred_perch + (1 - w) * best_ens_pred
#     a = kaggle_macro_auc(y_val, blend)
#     flag = '  ←' if a > best_auc else ''
#     print(f"{w:>8.2f} {1-w:>8.2f} {a:>12.4f}{flag}")
#     if a > best_auc:
#         best_auc, best_w = a, w
# print(f"\nBest blend: perch={best_w:.2f}  mlp_ensemble={1-best_w:.2f}  →  kaggle_auc={best_auc:.4f}")


# ─────────────────────────────────────────────────────────────────────────
# Stage 10 — Diverse head ensemble
# Train two extra small heads (linear probe + shallow MLP) on the lenient
# pseudo-augmented set, then ensemble with the existing residual head.
# ─────────────────────────────────────────────────────────────────────────

import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ── 1. Define two new lightweight heads ─────────────────────────────────
class LinearHead(nn.Module):
    """BN → Linear. Minimal, robust, hard to overfit."""
    def __init__(self, emb_dim=1536, n_classes=234):
        super().__init__()
        self.bn = nn.BatchNorm1d(emb_dim)
        self.fc = nn.Linear(emb_dim, n_classes)
    def forward(self, x):
        return self.fc(self.bn(x))   # logits


class ShallowMLPHead(nn.Module):
    """BN → 512 → GELU → Dropout → 234. One hidden layer, no residuals."""
    def __init__(self, emb_dim=1536, n_classes=234, hidden=512, dropout=0.30):
        super().__init__()
        self.bn = nn.BatchNorm1d(emb_dim)
        self.fc1 = nn.Linear(emb_dim, hidden)
        self.ln = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, n_classes)
    def forward(self, x):
        x = self.bn(x)
        x = F.gelu(self.ln(self.fc1(x)))
        x = self.drop(x)
        return self.fc2(x)   # logits


# ── 2. Pick which pseudo-set to train on ────────────────────────────────
# Lenient gave the best AUC last round — go with it.
PROFILE = 'lenient'
npz = np.load(AUTOSAVE_DIR / f'pseudo_labels_{PROFILE}.npz')
X_p, y_p = npz['X_pseudo'].astype(np.float32), npz['y_pseudo'].astype(np.float32)

X_all = np.concatenate([X_train, X_p], axis=0)
y_all = np.concatenate([y_train, y_p], axis=0)
rng = np.random.default_rng(42)
perm = rng.permutation(len(X_all))
X_all, y_all = X_all[perm], y_all[perm]

n_val = int(0.1 * len(X_all))
X_trn = torch.from_numpy(X_all[n_val:]).float()
y_trn = torch.from_numpy(y_all[n_val:]).float()
X_vld = torch.from_numpy(X_all[:n_val]).float()
y_vld = torch.from_numpy(y_all[:n_val]).float()

# Reuse the per-class pos_weight already computed in Stage 5
def weighted_bce(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw_t)


# ── 3. Generic train loop ───────────────────────────────────────────────
def train_head(model_cls, name, epochs=30, lr=1e-3, batch_size=256, patience=5):
    print(f"\n🎯 Training {name}")
    model = model_cls().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"   params: {n_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6
    )
    trn_loader = DataLoader(TensorDataset(X_trn, y_trn), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_vld, y_vld), batch_size=batch_size, shuffle=False)

    best_val, best_state, patience_cnt = float('inf'), None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        tl_losses = []
        for xb, yb in trn_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = weighted_bce(model(xb), yb)
            loss.backward()
            optimizer.step()
            tl_losses.append(loss.item())

        model.eval()
        vl_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vl_losses.append(weighted_bce(model(xb), yb).item())

        tl, vl = float(np.mean(tl_losses)), float(np.mean(vl_losses))
        print(f"   [Epoch {ep+1:2d}] loss={tl:.4f}  val={vl:.4f}  lr={optimizer.param_groups[0]['lr']:.1e}", flush=True)
        scheduler.step(vl)

        if vl < best_val - 1e-4:
            best_val, best_state, patience_cnt = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_cnt += 1
        if patience_cnt >= patience:
            print(f"   Early stop at epoch {ep+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"   Best val={best_val:.4f}  ({time.time()-t0:.1f}s)")
    out = AUTOSAVE_DIR / f'head_{name}_{PROFILE}.pt'
    torch.save(model.state_dict(), out)
    print(f"   💾 {out.name}")
    return model


linear_model  = train_head(LinearHead,     'linear',  epochs=30, lr=1e-3)
shallow_model = train_head(ShallowMLPHead, 'shallow', epochs=30, lr=8e-4)


# ── 4. Get predictions from all four sources ────────────────────────────
def predict_logits(model, X):
    model.eval()
    with torch.inference_mode():
        return model(torch.from_numpy(X).float().to(device)).cpu().numpy()

def sig(x):
    return 1.0 / (1.0 + np.exp(-x))

# Existing residual head (lenient version) — corrected kwargs
residual_model = ResidualHeadLogits(emb_dim=EMB_DIM, n_classes=n_species).to(device)
residual_model.load_state_dict(torch.load(AUTOSAVE_DIR / f'head_perch_pseudo_{PROFILE}.pt', map_location=device))

pred_lin   = sig(predict_logits(linear_model,   X_val))
pred_shal  = sig(predict_logits(shallow_model,  X_val))
pred_res   = sig(predict_logits(residual_model, X_val))
pred_perch = (1.0 / (1.0 + np.exp(-S_val))).astype(np.float32)


# ── 5. Score individuals + ensemble combinations ────────────────────────
def score(name, pred):
    a_ge3, k3, _ = macro_auc_ge3(y_val, pred)
    a_kg = kaggle_macro_auc(y_val, pred)
    print(f"  {name:<32} ge3={a_ge3:.4f}  kaggle={a_kg:.4f}  k3={k3}")
    return a_kg

print("\n" + "=" * 72)
print("INDIVIDUAL HEADS")
print("=" * 72)
score('direct Perch',           pred_perch)
score('linear',                 pred_lin)
score('shallow MLP',            pred_shal)
score('residual (lenient)',     pred_res)

print("\n" + "=" * 72)
print("ENSEMBLES (equal-weight average of MLP heads)")
print("=" * 72)
ens_lin_shal      = (pred_lin + pred_shal) / 2
ens_lin_res       = (pred_lin + pred_res) / 2
ens_shal_res      = (pred_shal + pred_res) / 2
ens_all_three     = (pred_lin + pred_shal + pred_res) / 3
score('lin + shal',             ens_lin_shal)
score('lin + res',              ens_lin_res)
score('shal + res',             ens_shal_res)
score('lin + shal + res',       ens_all_three)


# ── 6. Best-MLP-ensemble × direct Perch sweep ───────────────────────────
ensemble_candidates = {
    'lin + shal':         ens_lin_shal,
    'lin + res':          ens_lin_res,
    'shal + res':         ens_shal_res,
    'lin + shal + res':   ens_all_three,
}
best_ens_name = max(ensemble_candidates, key=lambda k: kaggle_macro_auc(y_val, ensemble_candidates[k]))
best_ens_pred = ensemble_candidates[best_ens_name]
print(f"\nBest MLP ensemble: {best_ens_name}")

print(f"\nBlend sweep: direct Perch ⨯ ({best_ens_name})")
print(f"{'perch_w':>8} {'mlp_w':>8} {'kaggle_auc':>12}")
best_auc, best_w = -1, 0.5
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * pred_perch + (1 - w) * best_ens_pred
    a = kaggle_macro_auc(y_val, blend)
    flag = '  ←' if a > best_auc else ''
    print(f"{w:>8.2f} {1-w:>8.2f} {a:>12.4f}{flag}")
    if a > best_auc:
        best_auc, best_w = a, w
print(f"\nBest blend: perch={best_w:.2f}  mlp_ensemble={1-best_w:.2f}  →  kaggle_auc={best_auc:.4f}")


🎯 Training linear
   params: 362,730
   [Epoch  1] loss=0.1463  val=0.0655  lr=1.0e-03
   [Epoch  2] loss=0.0509  val=0.0558  lr=1.0e-03
   [Epoch  3] loss=0.0402  val=0.0514  lr=1.0e-03
   [Epoch  4] loss=0.0343  val=0.0491  lr=1.0e-03
   [Epoch  5] loss=0.0303  val=0.0485  lr=1.0e-03
   [Epoch  6] loss=0.0276  val=0.0475  lr=1.0e-03
   [Epoch  7] loss=0.0254  val=0.0480  lr=1.0e-03
   [Epoch  8] loss=0.0236  val=0.0478  lr=1.0e-03
   [Epoch  9] loss=0.0222  val=0.0485  lr=1.0e-03
   [Epoch 10] loss=0.0188  val=0.0475  lr=5.0e-04
   [Epoch 11] loss=0.0178  val=0.0482  lr=5.0e-04
   Early stop at epoch 11
   Best val=0.0475  (23.0s)
   💾 head_linear_lenient.pt

🎯 Training shallow
   params: 911,082
   [Epoch  1] loss=0.1146  val=0.0691  lr=8.0e-04
   [Epoch  2] loss=0.0573  val=0.0538  lr=8.0e-04
   [Epoch  3] loss=0.0435  val=0.0466  lr=8.0e-04
   [Epoch  4] loss=0.0353  val=0.0435  lr=8.0e-04
   [Epoch  5] loss=0.0302  val=0.0409  lr=8.0e-04
   [Epoch  6] loss=0.0264  val=0.0410  lr